# SAC Training: Standard SAC with ActorMLP

Этот ноутбук обучает **SAC-агента (ActorMLP)** управлять камерой для Next-Best-View.

**Ключевая идея:** ActorMLP — и behavioral, и learned политика (стандартный SAC, без mismatch).  
ODIN NBV Head запускается на каждом шаге только как **источник признака `nbv_hint`** в `obs_vec[15:18]`.  
Coverage Head даёт сигнал награды: бинарный классификатор «есть ли скрытые объекты».

**Датасет сцен НЕ нужен** — среда PyBullet генерирует сцены динамически.

**Kaggle Inputs (подключить перед запуском):**
- Веса ODIN: загрузить `model_final.pth` как Kaggle Dataset (например `nbv-odin-weights`)
- *(Опционально)* Предыдущие checkpoints для resume

**Порядок запуска:** ячейки 1 → 2 → 3 → 4

## 1. Установка зависимостей (venv + ODIN стек)

In [1]:
import os
import subprocess
import sys
import urllib.request

CONFIG = {
    "ODIN_DIR": "my_odin",
    "ODIN_REPO_URL": "https://github.com/SergKurchev/my_odin.git",
    "ODIN_BRANCH": "feature/nbv_dataset_process",
    "RL_REPO_URL": "https://github.com/SergKurchev/article-nbv.git",
    "RL_DIR": "nbv_rl",
    "ODIN_WEIGHTS_URL": "https://huggingface.co/katefgroup/odin/resolve/main/scannet_resnet_47.8_73.3_32k_1.5k.pth",
    "ODIN_WEIGHTS_PATH": "my_odin/models/odin_scannet_context.pth",
    "M2F_WEIGHTS_URL": "https://huggingface.co/katefgroup/odin/resolve/main/m2f_coco.pkl",
    "M2F_WEIGHTS_PATH": "my_odin/models/model_final_5c90d4.pkl",
}

# Создаём venv
if not os.path.exists("venv"):
    subprocess.run(["apt-get", "update", "-y"], check=False)
    subprocess.run(["apt-get", "install", "-y", "python3.10", "python3.10-venv",
                    "python3.10-dev", "python3.10-distutils",
                    "libgl1", "libglib2.0-0"], check=False)
    subprocess.run(["python3.10", "-m", "venv", "venv", "--without-pip"], check=True)
    urllib.request.urlretrieve("https://bootstrap.pypa.io/get-pip.py", "get-pip.py")
    subprocess.run(["venv/bin/python", "get-pip.py"], check=True)
    os.remove("get-pip.py")
    print("venv created")

VENV_PYTHON = os.path.abspath("venv/bin/python")
VENV_PIP = os.path.abspath("venv/bin/pip")

def make_venv_env(extra=None):
    env = os.environ.copy()
    venv_dir = os.path.abspath("venv")
    env["VIRTUAL_ENV"] = venv_dir
    env["PATH"] = os.path.join(venv_dir, "bin") + ":" + env.get("PATH", "")
    env.pop("PYTHONPATH", None)
    if extra:
        env.update(extra)
    if "RL_DIR" in CONFIG:
        env["PYTHONPATH"] = os.path.abspath(CONFIG["RL_DIR"]) + ":" + os.path.abspath(CONFIG["ODIN_DIR"])
    return env

def run_cmd(cmd, cwd=None, env=None, check=True):
    if cmd[0] == "pip": cmd[0] = VENV_PIP
    elif cmd[0] == "python": cmd[0] = VENV_PYTHON
    print(f">>> {' '.join(str(c) for c in cmd)}")
    subprocess.run(cmd, cwd=cwd, env=env, check=check)

def clean_build(directory):
    import shutil, glob
    for d in ["build", "dist"]:
        path = os.path.join(directory, d)
        if os.path.exists(path): shutil.rmtree(path)
    for egg in glob.glob(os.path.join(directory, "*.egg-info")):
        shutil.rmtree(egg)

def install_all():
    venv_env = make_venv_env()
    cuda_env = make_venv_env({"FORCE_CUDA": "1", "TORCH_CUDA_ARCH_LIST": "6.0;7.0;7.5;8.0;8.6"})

    # 0. Клонируем ODIN
    if not os.path.exists(CONFIG["ODIN_DIR"]):
        subprocess.run(["git", "clone", "-q", "-b", CONFIG["ODIN_BRANCH"],
                        CONFIG["ODIN_REPO_URL"], CONFIG["ODIN_DIR"]], check=True)

    # 0b. Клонируем RL репозиторий (article-nbv)
    if not os.path.exists(CONFIG["RL_DIR"]):
        subprocess.run(["git", "clone", "-q", CONFIG["RL_REPO_URL"], CONFIG["RL_DIR"]], check=True)

    # 1. PyTorch 2.2.0 + CUDA 12.1
    print("\n1. Installing PyTorch...")
    run_cmd(["pip", "install", "-q", "torch==2.2.0", "torchvision==0.17.0",
             "--index-url", "https://download.pytorch.org/whl/cu121"], env=venv_env)
    run_cmd(["pip", "install", "-q", "torch-scatter",
             "-f", "https://data.pyg.org/whl/torch-2.2.0+cu121.html"], env=venv_env)

    # 2. NumPy + Pillow
    print("\n2. Installing NumPy + Pillow...")
    run_cmd(["pip", "install", "-q", "numpy<2", "--force-reinstall"], env=venv_env)
    run_cmd(["pip", "install", "-q", "Pillow>=10.2.0"], env=venv_env)

    # 3. Фильтрация requirements.txt
    print("\n3. Cleaning ODIN requirements...")
    req_path = os.path.join(CONFIG["ODIN_DIR"], "requirements.txt")
    with open(req_path, 'r') as f: lines = f.readlines()
    with open(req_path, 'w') as f:
        for line in lines:
            lc = line.strip().lower()
            if any(x in lc for x in ["waspinator", "detectron2", "pytorch3d"]): continue
            if "pyyaml==5.3.1" in lc: f.write("pyyaml>=5.4.1\n")
            else: f.write(line)

    # 4. Build tools
    print("\n4. Build tools...")
    run_cmd(["pip", "install", "-q", "cython", "setuptools", "wheel", "pycocotools"], env=venv_env)

    # 5. ODIN requirements
    print("\n5. ODIN requirements...")
    run_cmd(["pip", "install", "-q", "-r", req_path], env=venv_env)
    run_cmd(["pip", "install", "-q", "ninja", "fvcore", "iopath"], env=venv_env)

    # 6. Detectron2
    print("\n6. Detectron2...")
    run_cmd(["pip", "install", "-q", "--no-build-isolation",
             "git+https://github.com/facebookresearch/detectron2.git"], env=venv_env)

    # 7. PyTorch3D
    print("\n7. PyTorch3D...")
    run_cmd(["pip", "install", "-q", "--no-build-isolation",
             "git+https://github.com/facebookresearch/pytorch3d.git"], env=cuda_env)

    # 8. NumPy + OpenCV pin
    print("\n8. Pinning NumPy + OpenCV...")
    run_cmd(["pip", "uninstall", "-y", "-q", "numpy"], env=venv_env)
    run_cmd(["pip", "install", "-q", "numpy==1.26.4"], env=venv_env)
    run_cmd(["pip", "install", "-q", "opencv-python-headless==4.8.0.76"], env=venv_env)

    # 9. pointops2 CUDA kernel
    print("\n9. pointops2...")
    pointops_dir = os.path.abspath(os.path.join(CONFIG["ODIN_DIR"], "libs", "pointops2"))
    clean_build(pointops_dir)
    run_cmd(["python", "setup.py", "install"], cwd=pointops_dir, env=cuda_env)

    # 10. Deformable attention
    print("\n10. Deformable attention...")
    deform_dir = os.path.abspath(os.path.join(CONFIG["ODIN_DIR"], "odin", "modeling", "pixel_decoder", "ops"))
    clean_build(deform_dir)
    run_cmd(["python", "setup.py", "build", "install"], cwd=deform_dir, env=cuda_env)

    # 11. RL зависимости
    print("\n11. RL dependencies (pybullet, gymnasium)...")
    run_cmd(["pip", "install", "-q",
             "pybullet", "gymnasium",
             "imageio", "pandas", "matplotlib"], env=venv_env)

    print("\n=== Installation complete ===")

install_all()

Get:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,644 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]


Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [95.6 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://cli.github.com/packages stable/main amd64 Packages [356 B]


Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/multiverse amd64 Packages [77.8 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,945 kB]
Get:14 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,070 kB]
Get:15 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,295 kB]


Hit:16 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:17 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [3,018 kB]
Get:18 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:19 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease


Get:20 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages [7,347 kB]
Get:21 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy/main amd64 Packages [38.9 kB]
Get:22 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [4,292 kB]
Get:23 http://archive.ubuntu.com/ubuntu jammy-updates/multiverse amd64 Packages [86.4 kB]
Get:24 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,602 kB]
Get:25 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.2 MB]


...

[3/3] /usr/local/cuda/bin/nvcc --generate-dependencies-with-compile --dependency-output /kaggle/working/my_odin/odin/modeling/pixel_decoder/ops/build/temp.linux-x86_64-cpython-310/kaggle/working/my_odin/odin/modeling/pixel_decoder/ops/src/cuda/ms_deform_attn_cuda.o.d -DWITH_CUDA -I/kaggle/working/my_odin/odin/modeling/pixel_decoder/ops/src -I/kaggle/working/venv/lib/python3.10/site-packages/torch/include -I/kaggle/working/venv/lib/python3.10/site-packages/torch/include/torch/csrc/api/include -I/kaggle/working/venv/lib/python3.10/site-packages/torch/include/TH -I/kaggle/working/venv/lib/python3.10/site-packages/torch/include/THC -I/usr/local/cuda/include -I/kaggle/working/venv/include -I/usr/include/python3.10 -c -c /kaggle/working/my_odin/odin/modeling/pixel_decoder/ops/src/cuda/ms_deform_attn_cuda.cu -o /kaggle/working/my_odin/odin/modeling/pixel_decoder/ops/build/temp.linux-x86_64-cpython-310/kaggle/working/my_odin/odin/modeling/pixel_decoder/ops/src/cuda/ms_deform_attn_cuda.o -D__CU

running install
running install_lib
creating /kaggle/working/venv/lib/python3.10/site-packages/functions
copying build/lib.linux-x86_64-cpython-310/functions/__init__.py -> /kaggle/working/venv/lib/python3.10/site-packages/functions
copying build/lib.linux-x86_64-cpython-310/functions/ms_deform_attn_func.py -> /kaggle/working/venv/lib/python3.10/site-packages/functions
copying build/lib.linux-x86_64-cpython-310/MultiScaleDeformableAttention.cpython-310-x86_64-linux-gnu.so -> /kaggle/working/venv/lib/python3.10/site-packages
creating /kaggle/working/venv/lib/python3.10/site-packages/modules
copying build/lib.linux-x86_64-cpython-310/modules/__init__.py -> /kaggle/working/venv/lib/python3.10/site-packages/modules
copying build/lib.linux-x86_64-cpython-310/modules/ms_deform_attn.py -> /kaggle/working/venv/lib/python3.10/site-packages/modules
byte-compiling /kaggle/working/venv/lib/python3.10/site-packages/functions/__init__.py to __init__.cpython-310.pyc
byte-compiling /kaggle/working/ven

/kaggle/working/venv/lib/python3.10/site-packages/setuptools/_distutils/cmd.py:90: SetuptoolsDeprecationWarning: setup.py install is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` directly.
        Instead, use pypa/build, pypa/installer or other
        standards-based tools.

        This deprecation is overdue, please update your project and remove deprecated
        calls to avoid build errors in the future.

        See https://blog.ganssle.io/articles/2021/10/setup-py-deprecated.html for details.
        ********************************************************************************

!!
  self.initialize_options()



11. RL dependencies (pybullet, gymnasium)...
>>> /kaggle/working/venv/bin/pip install -q pybullet gymnasium imageio pandas matplotlib



=== Installation complete ===


## 2. Поиск весов ODIN (датасет сцен НЕ нужен)

In [2]:
ODIN_CFG = "my_odin/configs/scannet_context/3d.yaml"

# Ищем предобученные веса NBVActiveODIN
# Загрузите model_final.pth как Kaggle Dataset (напр. 'nbv-odin-weights')
POSSIBLE_WEIGHTS = [
    "/kaggle/input/notebooks/sergeistwpk/strawpick-segpoinnet-my-odin-nbv-2-active/output_nbv_stage2_active/model_final.pth",
    "/kaggle/input/nbv-odin-weights/model_final.pth",  # Kaggle Dataset
    "./output_nbv_stage2/model_final.pth",              # Из предыдущего запуска
    "./output_nbv_stage2/last_checkpoint.pth",
    "./output_odin_sac/best.pth",                       # Предыдущий SAC запуск
]

ODIN_WEIGHTS = None
for p in POSSIBLE_WEIGHTS:
    if os.path.exists(p):
        ODIN_WEIGHTS = p
        print(f"✓ Found ODIN weights: {p}")
        break

if ODIN_WEIGHTS is None:
    print("⚠ NBVActiveODIN weights not found! Downloading base ODIN weights...")
    ODIN_WEIGHTS = CONFIG["ODIN_WEIGHTS_PATH"]
    os.makedirs(os.path.dirname(ODIN_WEIGHTS), exist_ok=True)
    if not os.path.exists(ODIN_WEIGHTS):
        os.system(f"wget --tries=3 -q '{CONFIG['ODIN_WEIGHTS_URL']}' -O '{ODIN_WEIGHTS}'")

print(f"\nODIN_WEIGHTS = {ODIN_WEIGHTS}")
print(f"ODIN_CFG     = {ODIN_CFG}")
print(f"\nДатасет сцен НЕ нужен — PyBullet генерирует сцены динамически.")

✓ Found ODIN weights: /kaggle/input/notebooks/sergeistwpk/strawpick-segpoinnet-my-odin-nbv-2-active/output_nbv_stage2_active/model_final.pth

ODIN_WEIGHTS = /kaggle/input/notebooks/sergeistwpk/strawpick-segpoinnet-my-odin-nbv-2-active/output_nbv_stage2_active/model_final.pth
ODIN_CFG     = my_odin/configs/scannet_context/3d.yaml

Датасет сцен НЕ нужен — PyBullet генерирует сцены динамически.


## 3. Синхронизация checkpoint (для продолжения обучения)

In [3]:
import shutil

OUTPUT_DIR = "./output_odin_sac"

def sync_previous_output(target_output=OUTPUT_DIR):
    """Копирует старые checkpoints из /kaggle/input."""
    os.makedirs(target_output, exist_ok=True)
    for root, dirs, files in os.walk("/kaggle/input"):
        if "output_odin_sac" in root or "output_rl" in root:
            pth_files = [f for f in files if f.endswith(".pth")]
            if pth_files:
                print(f"Found previous checkpoint in: {root}")
                for f in pth_files:
                    src = os.path.join(root, f)
                    dst = os.path.join(target_output, f)
                    if not os.path.exists(dst):
                        shutil.copy2(src, dst)
                        print(f"  Copied: {f}")
                return
    print("No previous checkpoints found. Training from scratch.")

sync_previous_output()

# Проверяем наличие best/last для информации
for f in ["best.pth", "last.pth"]:
    path = os.path.join(OUTPUT_DIR, f)
    if os.path.exists(path):
        print(f"✓ Found: {path}")

No previous checkpoints found. Training from scratch.


## 4. Запуск обучения: Standard SAC

ActorMLP — и behavioral policy (сбор данных), и learned policy (SAC-обновления).  
ODIN NBV Head заморожен; его выход передаётся как `nbv_hint` в obs — ActorMLP его видит.

**`--freeze_backbone`** замораживает весь ODIN backbone.  
Обучаются: **ActorMLP + Twin Q-critics + log\_std + log\_alpha**.

In [4]:
!cd /kaggle/working/my_odin && git pull


Already up to date.


In [5]:
subprocess.run([
        "git", "rev-parse", "--short", "HEAD"
    ], cwd = "/kaggle/working/my_odin")

260ba01


CompletedProcess(args=['git', 'rev-parse', '--short', 'HEAD'], returncode=0)

In [6]:
!cd nbv_rl && git pull origin master


From https://github.com/SergKurchev/article-nbv
 * branch              master     -> FETCH_HEAD
Already up to date.


In [7]:
subprocess.run([
        "git", "rev-parse", "--short", "HEAD"
    ], cwd = "/kaggle/working/nbv_rl")

2717feb0


CompletedProcess(args=['git', 'rev-parse', '--short', 'HEAD'], returncode=0)

In [8]:
RL_DIR = os.path.abspath(CONFIG["RL_DIR"])
ODIN_DIR = os.path.abspath(CONFIG["ODIN_DIR"])

# ====================================================================
# Конфигурация обучения — редактируйте параметры здесь
# ====================================================================
TOTAL_STEPS      = 200000   # Общее число шагов SAC
LR_ACTOR         = "1e-4"   # Learning rate ActorMLP (actor)
LR_CRITIC        = "3e-4"   # Learning rate Q-networks (critic)
BUFFER_SIZE      = "100000"  # Replay Buffer (первые LEARNING_STARTS шагов — случайное заполнение)
GRADIENT_STEPS   = "4"      # SAC-апдейтов за 1 шаг среды (UTD ratio, ~4x GPU утилизация)
BATCH_SIZE       = "512"     # Batch для SAC updates
LEARNING_STARTS  = "1000"    # Шагов до начала SAC updates
SCENE_STAGE      = "2"      # 1=single obj, 2=multi obj, 3=multi+obstacles
FREEZE_BACKBONE  = True     # True (рекомендуется): backbone заморожен, обучается ActorMLP + critics
TRAIN_LAST_TRANSFORMER_BLOCK = False  # Дополнительно разморозить последний блок трансформера ODIN
# ====================================================================

train_cmd = [
    VENV_PYTHON,
    f"{RL_DIR}/train_odin_sac_rl.py",

    # --- Веса и конфиг ODIN ---
    "--odin_weights", ODIN_WEIGHTS,
    "--odin_cfg", ODIN_CFG,

    # --- Параметры SAC ---
    "--total_steps", str(TOTAL_STEPS),
    "--lr_actor", LR_ACTOR,
    "--lr_critic", LR_CRITIC,
    "--buffer_size", BUFFER_SIZE,
    "--batch_size", BATCH_SIZE,
    "--learning_starts", LEARNING_STARTS,
    "--gradient_steps", GRADIENT_STEPS,

    # --- Конфигурация сцены ---
    "--scene_stage", SCENE_STAGE,
    "--num_classes", "24",

    # --- Output ---
    "--output_dir", OUTPUT_DIR,
    "--save_freq", "5000",
    "--log_freq", "10",

]

if FREEZE_BACKBONE:
    train_cmd.append("--freeze_backbone")
if TRAIN_LAST_TRANSFORMER_BLOCK:
    train_cmd.append("--train_last_transformer_block")

venv_env = make_venv_env({
    "PYTHONPATH": f"{RL_DIR}:{ODIN_DIR}",
    "PYBULLET_HEADLESS": "1",
    "DISPLAY": "",
})

print("=" * 60)
print("  SAC Training: ActorMLP = behavioral + learned policy")
print(f"  ODIN weights: {ODIN_WEIGHTS}")
print(f"  Total steps:  {TOTAL_STEPS}")
print(f"  Freeze:       {FREEZE_BACKBONE}")
print(f"  Scene stage:  {SCENE_STAGE}")
print(f"  Buffer: {BUFFER_SIZE}  Batch: {BATCH_SIZE}  Starts: {LEARNING_STARTS}")
print("=" * 60)
print()

subprocess.run(train_cmd, env=venv_env, check=True)


  SAC Training: ActorMLP = behavioral + learned policy
  ODIN weights: /kaggle/input/notebooks/sergeistwpk/strawpick-segpoinnet-my-odin-nbv-2-active/output_nbv_stage2_active/model_final.pth
  Total steps:  200000
  Freeze:       True
  Scene stage:  2
  Buffer: 100000  Batch: 512  Starts: 1000



[transformers] Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.0+cu121


[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


/kaggle/working/venv/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Skipped 2 size-mismatched keys (will use random init): ['coverage_head.mlp.0.weight', 'nbv_head.trunk.0.weight']
Weight format of ODINHead have changed! Please upgrade your models. Applying automatic conversion now ...
Missing keys (180): ['model.sem_seg_head.pixel_decoder.input_proj.0.0.weight', 'model.sem_seg_head.pixel_decoder.input_proj.0.0.bias', 'model.sem_seg_head.pixel_decoder.input_proj.0.1.weight', 'model.sem_seg_head.pixel_decoder.input_proj.0.1.bias', 'model.sem_seg_head.pixel_decoder.input_proj.1.0.weight']...
Unexpected keys (178): ['model.sem_seg_head.pixel_decoder.pixel_decoder.input_proj.0.0.weight', 'model.sem_seg_head.pixel_decoder.pixel_decoder.input_proj.0.0.bias', 'model.sem_seg_head.pixel_decoder.pixel_decoder.input_proj.0.1.weight', 'model.sem_seg_head.pixel_decoder.pixel_decoder.input_proj.0.1.bias', 'model.sem_seg_head.pixel_decoder.pixel_decoder.input_proj.1.0.weight']...
pybullet build time: Jan 29 2025 23:16:28


...

Number of frames: 5
Number of frames: 1
Number of frames: 1
Number of frames: 2
Number of frames: 2
Number of frames: 3
Number of frames: 3
Number of frames: 4
Number of frames: 4
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5


Number of frames: 1
Number of frames: 1
Number of frames: 2
Number of frames: 2
Number of frames: 3
Number of frames: 3
Number of frames: 4
Number of frames: 4
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5


Number of frames: 1
Number of frames: 2
Number of frames: 2
Number of frames: 3
Number of frames: 3
Number of frames: 4
Number of frames: 4
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 1


[Step  86500|Ep 3460] R= -75.00  avg= -22.96  p=0.894  col=1.00  oob=0.00  α=0.000  cL=25765654166340215242752.000  aL=-136339120128.000  covL=0.112  gpuMem=0.2GB
Number of frames: 1
Number of frames: 1
Number of frames: 2
Number of frames: 2
Number of frames: 3
Number of frames: 3
Number of frames: 4
Number of frames: 4
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames:

Number of frames: 1
Number of frames: 2
Number of frames: 2
Number of frames: 3
Number of frames: 3
Number of frames: 4
Number of frames: 4
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 5
Number of frames: 1


## Результаты

Файлы сохраняются в `./output_odin_sac/`:

| Файл | Описание |
|------|----------|
| `best.pth` | Лучшая политика (по суммарной награде за эпизод) |
| `last.pth` | Последний checkpoint |
| `ckpt_step*.pth` | Промежуточные checkpoints (каждые 5000 шагов) |
| `sac_metrics.csv` | Метрики: reward, p_hidden, critic/actor loss, alpha |
| `logs/training_metrics.csv` | Пошаговые метрики среды |

### Что содержит checkpoint (.pth)
```python
state = torch.load('best.pth')
state['actor_mlp']  # Веса ActorMLP (= политика, и behavioral и learned)
state['nbv_head']   # Веса NBV Head + Coverage Head (заморожены, сохраняются для inference)
state['critic']     # Twin Q-networks
state['log_std']    # Learnable exploration noise [6]
state['log_alpha']  # Entropy coefficient α
```

### Архитектура
```
ODIN Backbone (frozen) → scene_emb [256]
                               |
           NBV Head (frozen) → nbv_hint → obs_vec[15:18]
                               |
    obs_vec_full [274] = [v_t || scene_emb]
                               |
      ActorMLP (Actor) → next_camera_pose   ← обучается SAC (behavioral + learned)
      Coverage Head    → p_hidden           ← сигнал награды, обучается BCE
      Twin Q-Critics   → Q(s, a)            ← обучается SAC
```

### Первые ~20 эпизодов
ActorMLP стартует из случайной инициализации — ожидаются коллизии и OOB.  
Это нормально: Q быстро обучается штрафовать плохие действия, политика стабилизируется.